# 20 · Fix shared-point locations with validated name matches (Part D, step D3b)

**Problem (notebook 18):** 18,665 India schools and 13,289 health facilities share one point with 3 or more others —
village / cluster medians or default points. The real building is unknown.

**Method:** look for a place **with the same name** near the current point in an independent source (notebook 19:
OpenStreetMap schools and health places; Overture Maps Places), using a **Sedona broadcast distance join** for the
candidates and the project's name cleaning (`core()`) + rapidfuzz for the names. A new point is accepted only if
the name score is ≥ 95, no second candidate scores as high, the place is claimed by no other facility (and, for
schools, not already used in Part B), and the facility's name is unique in its village / district.

**Accuracy first — validation on Kerala, where the truth is known:** schools are anchored at their village's
median UDISE point and compared with the verified Kerala layer; health facilities are anchored at their health
block's median and compared with the official DHS Kerala 2024-25 coordinates. **A source is used only if ≥ 90 %
of its accepted matches lie within 250 m of the truth.** (Prototype result: OSM passes for schools and health;
Overture does not, so it is reported but not used.)

**Output:** the two final CSVs are updated **only for validated matches** (Kerala rows never change); the previous
point is kept in `pre_d3b_lat/lon`, with `d3b_place_id`, `d3b_place_name`, `d3b_name_score`, `d3b_moved_m`. The old
files are copied to `data/interim/backup_before_d3b/` first. `gold/d3b_validation.csv`, `gold/d3b_fixes_summary.csv`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import datetime as dt
import json
import shutil

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from bigdata.spark_session import get_spark
from build_india_schools import core as school_core, load_osm as load_osm_schools  # Part B helpers
from build_india_health_facilities import core as health_core, haversine_m

FINAL = PROJECT / "FINAL COMPLETE DATASET"
SCHOOLS_CSV = FINAL / "schools" / "india_schools_govt_aided.csv"
HEALTH_CSV = FINAL / "health" / "india_health_facilities_final.csv"
OSM_DIR = PROJECT / "data" / "raw" / "osm" / "india"
OV_FILE = PROJECT / "data" / "raw" / "overture" / "overture_places_india_2026-10-21.0.parquet"
GOLD = PROJECT / "data" / "lake" / "gold"
BACKUP = PROJECT / "data" / "interim" / "backup_before_d3b"
spark = get_spark(PROJECT, cores=6, memory="6g")
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

cores: 6 | Spark UI: http://127.0.0.1:4040


## Candidate places

In [5]:
# Candidate places. OSM schools: the Part B loader (name:en preferred, usable core name). OSM health: notebook 19.
# Overture (notebook 19): loaded only to document the validation; it is not used for fixes unless it passes.
osm_s = load_osm_schools().rename(columns={"osm_id": "pid", "osm_name": "pname", "osm_lat": "lat", "osm_lon": "lon", "ocore": "pcore"})

rows = []
for f in sorted(OSM_DIR.glob("osm_health_*.json")):
    for e in json.loads(f.read_text(encoding="utf-8"))["elements"]:
        t = e.get("tags", {})
        name = t.get("name:en") or t.get("name") or ""
        lat, lon = e.get("lat", e.get("center", {}).get("lat")), e.get("lon", e.get("center", {}).get("lon"))
        if name and lat is not None:
            rows.append((f"{e['type']}/{e['id']}", name, lat, lon))
osm_h = pd.DataFrame(rows, columns=["pid", "pname", "lat", "lon"]).drop_duplicates("pid")
osm_h["pcore"] = osm_h["pname"].map(health_core)
osm_h = osm_h[osm_h["pcore"].str.len() >= 4].reset_index(drop=True)

ov = pd.read_parquet(OV_FILE).dropna(subset=["name"])
OV_HEALTH = {"hospital", "health_care", "outpatient_care_facility", "medical_service", "primary_care_or_general_clinic",
             "emergency_or_urgent_care_facility", "emergency_department", "specialty_hospital", "walk_in_clinic",
             "specialized_medical_facility"}
ov_s = ov[ov["taxonomy_path"].str.startswith("education>place_of_learning>school") & (ov["basic_category"] != "preschool")]
ov_h = ov[ov["basic_category"].isin(OV_HEALTH)]
ov_s = ov_s.assign(pid="overture/" + ov_s["id"], pname=ov_s["name"], pcore=ov_s["name"].map(school_core))[["pid", "pname", "lat", "lon", "pcore"]]
ov_h = ov_h.assign(pid="overture/" + ov_h["id"], pname=ov_h["name"], pcore=ov_h["name"].map(health_core))[["pid", "pname", "lat", "lon", "pcore"]]
ov_s, ov_h = [d[d["pcore"].str.len() >= 4].reset_index(drop=True) for d in (ov_s, ov_h)]
print(f"OSM schools {len(osm_s):,} | OSM health {len(osm_h):,} | Overture schools {len(ov_s):,} | Overture health {len(ov_h):,}")

OSM: 7 files, 33694 named schools, 31048 with a usable name


OSM schools 31,048 | OSM health 130,620 | Overture schools 206,322 | Overture health 156,744


## Matching rules

In [6]:
def candidates(targets, pois, radius_m):
    """Sedona broadcast distance join (the fastest strategy in notebook 21): every (target, place) pair within radius_m."""
    t = spark.createDataFrame(targets[["tid", "a_lat", "a_lon"]]).selectExpr("tid", "ST_Point(a_lon, a_lat) AS g")
    p = spark.createDataFrame(pois[["pid", "lat", "lon"]]).selectExpr("pid", "ST_Point(lon, lat) AS g")
    t.createOrReplaceTempView("t")
    p.createOrReplaceTempView("p")
    return spark.sql(f"SELECT /*+ BROADCAST(p) */ t.tid, p.pid FROM t JOIN p ON ST_DWithin(t.g, p.g, {radius_m}, true)").toPandas()


def match(targets, pois, radius_m, min_score, used=frozenset()):
    """Accept a place only if: same core name (score >= min_score), no second place with such a score, the place is
    claimed by no other target, it is not already used by another facility, and the target's core name is unique in its
    village / district (no twins such as two 'Rampur' schools)."""
    pairs = candidates(targets, pois, radius_m)
    if pairs.empty:
        return pairs.assign(score=[])
    pairs = pairs.merge(targets[["tid", "c"]], on="tid").merge(pois[["pid", "pcore"]], on="pid")
    pairs["score"] = [fuzz.ratio(a, b) for a, b in zip(pairs["c"], pairs["pcore"])]
    pairs = pairs.sort_values(["tid", "score"], ascending=[True, False])
    pairs["rank"] = pairs.groupby("tid").cumcount()
    best = pairs[pairs["rank"] == 0].set_index("tid")
    second = pairs[pairs["rank"] == 1].set_index("tid")["score"].reindex(best.index).fillna(0)
    a = best[(best["score"] >= min_score) & (second < min_score)].reset_index()
    a = a[~a["pid"].duplicated(keep=False) & ~a["pid"].isin(used)]
    a = a[a["tid"].isin(targets.loc[targets["twin_ok"], "tid"])]
    return a[["tid", "pid", "score"]]


def evaluate(label, targets, pois, radius_m, min_score):
    """Validation against true locations (columns t_lat/t_lon): share of accepted matches within 250 m / 500 m."""
    a = match(targets, pois, radius_m, min_score).merge(targets[["tid", "t_lat", "t_lon"]], on="tid")                                                 .merge(pois[["pid", "lat", "lon"]], on="pid")
    err = haversine_m(a["lat"], a["lon"], a["t_lat"], a["t_lon"]) if len(a) else np.array([])
    r = {"test": label, "radius_km": radius_m / 1000, "min_score": min_score, "targets": len(targets), "accepted": len(a),
         "share_accepted": round(len(a) / len(targets), 4),
         "within_250m": round(float(np.mean(err <= 250)), 3) if len(a) else None,
         "within_500m": round(float(np.mean(err <= 500)), 3) if len(a) else None,
         "median_error_m": round(float(np.median(err))) if len(a) else None}
    print(r)
    return r

## Validation on Kerala (decides which sources are used)

In [7]:
# Validation on Kerala, where the true locations are known.
# Schools: anchor = median of the UDISE points of the school's village (how a shared village point is made),
#          truth = the verified Kerala layer.  Health: anchor = median of the facilities of the health block,
#          truth = the official DHS Kerala 2024-25 coordinates.
s = pd.read_csv(SCHOOLS_CSV, dtype=str)  # all text: the file is written back unchanged except the fixed rows
for c in ["lat", "lon", "udise_lat", "udise_lon"]:
    s[c + "_n"] = pd.to_numeric(s[c])
k = s[(s["state_name"] == "Kerala") & s["udise_lat_n"].notna() & s["lat_n"].notna()].copy()
k[["a_lat", "a_lon"]] = k.groupby("village_code")[["udise_lat_n", "udise_lon_n"]].transform("median")
k = k.assign(tid=np.arange(len(k)), c=k["school_name"].map(school_core), t_lat=k["lat_n"], t_lon=k["lon_n"])
k["twin_ok"] = k.groupby(["village_code", "c"])["tid"].transform("size").eq(1) & k["c"].str.len().ge(4)

dhs = pd.read_csv(FINAL / "health" / "kerala_health_facilities_final.csv", low_memory=False)
d = dhs[(dhs["location_from"] == "DHS_2024_25") & dhs["lat"].notna() & dhs["health_block"].notna()].copy()
d[["a_lat", "a_lon"]] = d.groupby("health_block")[["lat", "lon"]].transform("median")
d = d.assign(tid=np.arange(len(d)), c=d["name"].map(health_core), t_lat=d["lat"], t_lon=d["lon"])
d["twin_ok"] = d.groupby(["health_block", "c"])["tid"].transform("size").eq(1) & d["c"].str.len().ge(4)

SETTINGS = {"schools": (3000, 95), "health": (10000, 95)}  # chosen on the prototype runs (D3b notes in the plan)
validation = pd.DataFrame([
    evaluate("Kerala schools vs OSM", k, osm_s, *SETTINGS["schools"]),
    evaluate("Kerala schools vs Overture", k, ov_s, *SETTINGS["schools"]),
    evaluate("Kerala health (DHS) vs OSM", d, osm_h, *SETTINGS["health"]),
    evaluate("Kerala health (DHS) vs Overture", d, ov_h, *SETTINGS["health"]),
])
validation["passes_90pct_rule"] = validation["within_250m"].fillna(0) >= 0.90
validation.to_csv(GOLD / "d3b_validation.csv", index=False)
print(validation[["test", "accepted", "within_250m", "within_500m", "median_error_m", "passes_90pct_rule"]].to_string(index=False))
USE = {"schools": [("osm", osm_s)] if validation.loc[0, "passes_90pct_rule"] else [],
       "health": [("osm", osm_h)] if validation.loc[2, "passes_90pct_rule"] else []}
for kind, i in [("schools", 1), ("health", 3)]:  # Overture is used only if it passes too
    if validation.loc[i, "passes_90pct_rule"]:
        USE[kind].append(("overture", ov_s if kind == "schools" else ov_h))
print("sources used:", {kind: [n for n, _ in v] for kind, v in USE.items()})

{'test': 'Kerala schools vs OSM', 'radius_km': 3.0, 'min_score': 95, 'targets': 11376, 'accepted': 712, 'share_accepted': 0.0626, 'within_250m': 0.89, 'within_500m': 0.924, 'median_error_m': 23}


{'test': 'Kerala schools vs Overture', 'radius_km': 3.0, 'min_score': 95, 'targets': 11376, 'accepted': 387, 'share_accepted': 0.034, 'within_250m': 0.61, 'within_500m': 0.685, 'median_error_m': 104}


{'test': 'Kerala health (DHS) vs OSM', 'radius_km': 10.0, 'min_score': 95, 'targets': 1046, 'accepted': 101, 'share_accepted': 0.0966, 'within_250m': 0.95, 'within_500m': 0.97, 'median_error_m': 32}


{'test': 'Kerala health (DHS) vs Overture', 'radius_km': 10.0, 'min_score': 95, 'targets': 1046, 'accepted': 27, 'share_accepted': 0.0258, 'within_250m': 0.704, 'within_500m': 0.741, 'median_error_m': 26}
                           test  accepted  within_250m  within_500m  median_error_m  passes_90pct_rule
          Kerala schools vs OSM       712        0.890        0.924              23              False
     Kerala schools vs Overture       387        0.610        0.685             104              False
     Kerala health (DHS) vs OSM       101        0.950        0.970              32               True
Kerala health (DHS) vs Overture        27        0.704        0.741              26              False
sources used: {'schools': [], 'health': ['osm']}


## India schools

In [8]:
# India schools (Kerala rows are never touched): targets = points shared by 4+ schools or village / cluster medians
s["tid"] = np.arange(len(s))
s["c"] = s["school_name"].map(school_core)
shared = s.groupby(["lat_n", "lon_n"])["tid"].transform("size").ge(4)
s["twin_ok"] = s.groupby(["village_code", "c"])["tid"].transform("size").eq(1) & s["c"].str.len().ge(4)
tgt = s[(s["state_name"] != "Kerala") & s["lat_n"].notna() &
        (shared | s["location_source"].isin(["village_median", "cluster_median"]))].assign(a_lat=lambda x: x["lat_n"], a_lon=lambda x: x["lon_n"])
print(f"school targets: {len(tgt):,}")
used = set(s["osm_id"].dropna())  # OSM schools already given to a school in Part B
fix_s = []
for name, pois in USE["schools"]:
    a = match(tgt[~tgt["tid"].isin([r for f in fix_s for r in f["tid"]])], pois, *SETTINGS["schools"], used=frozenset(used))
    fix_s.append(a.merge(pois[["pid", "pname", "lat", "lon"]], on="pid").assign(src=name))
    used |= set(a["pid"])
fix_s = pd.concat(fix_s) if fix_s else pd.DataFrame(columns=["tid", "pid", "pname", "lat", "lon", "score", "src"])
print(f"schools fixed: {len(fix_s):,} of {len(tgt):,} targets ({len(fix_s) / len(tgt):.1%})")

school targets: 27,693
schools fixed: 0 of 27,693 targets (0.0%)


## India health facilities

In [9]:
# India health (Kerala rows are never touched): targets = points shared by 4+ facilities or flagged placeholder / low precision
h = pd.read_csv(HEALTH_CSV, low_memory=False, dtype=str)
for c in ["lat", "lon"]:
    h[c + "_n"] = pd.to_numeric(h[c])
h["tid"] = np.arange(len(h))
h["c"] = h["name"].map(health_core)
shared_h = h.groupby(["lat_n", "lon_n"])["tid"].transform("size").ge(4)
flagged = h["qa_flags"].fillna("").str.contains("placeholder_point|low_precision")
h["twin_ok"] = h.groupby(["state", "district", "c"])["tid"].transform("size").eq(1) & h["c"].str.len().ge(4)
tgt_h = h[(h["state"].str.lower() != "kerala") & h["lat_n"].notna() & (shared_h | flagged)].assign(a_lat=lambda x: x["lat_n"], a_lon=lambda x: x["lon_n"])
print(f"health targets: {len(tgt_h):,}")
fix_h, used_h = [], set()
for name, pois in USE["health"]:
    a = match(tgt_h[~tgt_h["tid"].isin([r for f in fix_h for r in f["tid"]])], pois, *SETTINGS["health"], used=frozenset(used_h))
    fix_h.append(a.merge(pois[["pid", "pname", "lat", "lon"]], on="pid").assign(src=name))
    used_h |= set(a["pid"])
fix_h = pd.concat(fix_h) if fix_h else pd.DataFrame(columns=["tid", "pid", "pname", "lat", "lon", "score", "src"])
print(f"health fixed: {len(fix_h):,} of {len(tgt_h):,} targets ({len(fix_h) / len(tgt_h):.1%})")

health targets: 20,468


health fixed: 910 of 20,468 targets (4.4%)


## Write the updated final files

In [10]:
# Write: back up the final files, then move only the validated rows; the previous point is kept in pre_d3b_lat/lon.
BACKUP.mkdir(parents=True, exist_ok=True)
for f in [SCHOOLS_CSV, HEALTH_CSV]:
    if not (BACKUP / f.name).exists():
        shutil.copy2(f, BACKUP / f.name)
today = dt.date.today().isoformat()


def apply(df, fixes, source_col):
    """Only rows with a validated match change (lat, lon, source); every other value stays the exact original text."""
    f = fixes.set_index("tid")
    i = df["tid"].isin(f.index)
    tid = df.loc[i, "tid"]
    new_lat, new_lon = tid.map(f["lat"]), tid.map(f["lon"])
    df["pre_d3b_lat"] = df["lat"].where(i)
    df["pre_d3b_lon"] = df["lon"].where(i)
    df["d3b_place_id"] = df["tid"].map(f["pid"])
    df["d3b_place_name"] = df["tid"].map(f["pname"])
    df["d3b_name_score"] = df["tid"].map(f["score"])
    df["d3b_moved_m"] = 0.0
    if i.any():  # no validated source for a layer -> nothing moves
        df.loc[i, "d3b_moved_m"] = np.round(haversine_m(df.loc[i, "lat_n"], df.loc[i, "lon_n"], new_lat, new_lon))
        df.loc[i, "lat"] = new_lat.map(lambda v: f"{v:.7f}")
        df.loc[i, "lon"] = new_lon.map(lambda v: f"{v:.7f}")
        df.loc[i, source_col] = tid.map(f["src"]) + "_name_match_d3b"
    return df.drop(columns=["tid", "c", "twin_ok", *[c for c in df.columns if c.endswith("_n")]])


def unchanged_except_fixed(old, new, cols_changed):
    """Every original column of every row is identical, except the listed columns on fixed rows."""
    fixed = new["d3b_place_id"].notna().to_numpy()
    keep = [c for c in old.columns if not c.endswith("_n") and c not in ("tid", "c", "twin_ok")]
    a, b = old[keep].fillna(""), new[keep].fillna("")
    same = (a.to_numpy() == b.to_numpy())
    same[fixed] |= np.isin(keep, cols_changed)[None, :]
    return bool(same.all())


s_out = apply(s.copy(), fix_s, "location_source")
s_out.loc[s_out["d3b_place_id"].notna(), "location_quality"] = "osm_matched"
h_out = apply(h.copy(), fix_h, "location_from")
h_out.loc[h_out["d3b_place_id"].notna(), "location_quality"] = "good"
assert len(s_out) == len(s) and len(h_out) == len(h)
assert unchanged_except_fixed(s, s_out, ["lat", "lon", "location_source", "location_quality"]), "unexpected change in schools"
assert unchanged_except_fixed(h, h_out, ["lat", "lon", "location_from", "location_quality"]), "unexpected change in health"
assert not s_out.loc[s_out["state_name"] == "Kerala", "d3b_place_id"].notna().any(), "a Kerala school was changed"
s_out.to_csv(SCHOOLS_CSV, index=False)
h_out.to_csv(HEALTH_CSV, index=False)
summary = pd.DataFrame([
    {"layer": "schools_india", "targets": len(tgt), "fixed": len(fix_s), "median_moved_m": s_out.loc[s_out["d3b_moved_m"].astype(float) > 0, "d3b_moved_m"].astype(float).median()},
    {"layer": "health_facilities", "targets": len(tgt_h), "fixed": len(fix_h), "median_moved_m": h_out.loc[h_out["d3b_moved_m"].astype(float) > 0, "d3b_moved_m"].astype(float).median()}])
summary["share_fixed"] = (summary["fixed"] / summary["targets"]).round(4)
summary.to_csv(GOLD / "d3b_fixes_summary.csv", index=False)
print(summary.to_string(index=False))
print("backups of the previous files:", BACKUP)
spark.stop()

            layer  targets  fixed  median_moved_m  share_fixed
    schools_india    27693      0             NaN       0.0000
health_facilities    20468    910          1343.5       0.0445
backups of the previous files: C:\Users\ASUS\Desktop\sem3\BIG SPATIAL DATA\PROJECT- SPATIAL INEQUALITY ASESSMENT\data\interim\backup_before_d3b
